# SPENPy simulation demo

This notebook walks through the **simulation + reconstruction** API of
`spenpy.spen.spen`, using a 2D brain image as the ground truth.

It complements the **`demo/`** folder, which is dedicated to
reconstructing real Bruker PV360 SPEN datasets (`pv360.m`-equivalent
pipeline). If you want to run the real-data pipeline, see
[`demo/README.md`](demo/README.md).

The simulator now supports YAML profiles. This demo uses
`spenpy/configs/scanner_like.yaml`, which is shaped by the included real
scanner data, and returns metadata with the sampled synthetic artifacts.

The notebook is organised as:

1. Load a grayscale image as the ground truth.
2. Build a YAML-configured `spen` simulator and obtain the encoding
   matrices `AFinal` / `InvA`.
3. Run a full simulation (`sim`) producing corrupted SPEN data, a phase
   map estimate, and a clean low-resolution reference.
4. Reconstruct using **the correct order: phase correction first, then
   `InvA`** — and contrast against the wrong order, which yields
   ghost-like artifacts.
5. Discuss why even the correctly-ordered reconstruction is not a
   pixel-perfect identity (`InvA` is a *weighted adjoint*, not a true
   matrix inverse).
6. Show the fast forward path that bypasses full simulation by applying
   `AFinal` directly.


In [ ]:
from pathlib import Path

from spenpy.spen import spen
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F


def find_spenpy_repo() -> Path:
    """Locate the outer spenpy repository from common notebook launch dirs."""
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / "spenpy/data/brain.png").exists():
            return base
        if (base / "spenpy/spenpy/data/brain.png").exists():
            return base / "spenpy"
    raise FileNotFoundError("Could not locate spenpy/data/brain.png from the current working directory")


def pil_to_float_tensor(image: Image.Image) -> torch.Tensor:
    image = image.convert("L")
    data = torch.frombuffer(bytearray(image.tobytes()), dtype=torch.uint8).to(torch.float32)
    return data.reshape(image.height, image.width) / 255.0


def to_plot(value: torch.Tensor) -> torch.Tensor:
    return value.detach().cpu()


SPENPY_REPO = find_spenpy_repo()
print("SPENPy repo:", SPENPY_REPO)


## 1. Load a grayscale image as the "true" object

We use `spenpy/data/brain.png` and keep just the first channel (grayscale).
This image plays the role of the spin density that the SPEN sequence
will encode.


In [ ]:
path = SPENPY_REPO / "spenpy/data/brain.png"
img = pil_to_float_tensor(Image.open(path))

print("img.shape:", tuple(img.shape))  # (256, 256)


## 2. Build the YAML-configured simulator and run a full simulation

`spen.from_yaml(...)` loads scanner geometry and artifact randomization
from a YAML file. In this notebook we use `scanner_like.yaml`, which is
based on the included Bruker PV360 SPEN scans:

- `PVM_Matrix` is mostly `96 x 96`.
- `PVM_FovCm` is often `1.6 x 1.6` cm.
- `NSegments` is `1` in the included SPEN scans.
- EPI echo spacing is in the `0.2304` to `0.384` ms range.

`spen.get_InvA()` returns `(InvA, AFinal)`:

- `AFinal` is the **forward** SR encoding matrix, shape `(PE, PE)`,
  `complex64`.
- `InvA` is the **weighted adjoint** matrix used as a fast approximate
  inverse. **Important:** `InvA` is *not* a true matrix inverse of
  `AFinal` — see Section 5 below.

`spen.sim(...)` returns:

- `corrupted_data`: complex `(B, PE, RO)` — simulated SPEN data after
  B0, shot phase, even/odd phase, trajectory, and noise artifacts.
- `phase_map`: float `(B, PE/2, RO)` — an estimate of the even-line
  phase corruption. It may intentionally differ from truth.
- `good_lr_image`: complex `(B, PE, RO)` — clean low-resolution SPEN
  data before artifact corruption.
- `sim_meta`: metadata containing the effective config, sampled artifact
  values, `phase_map_true`, and `shot_phase_map`.

The full YAML schema is documented in `docs/simulation_yaml.md`.


In [ ]:
config_path = SPENPY_REPO / "spenpy/configs/scanner_like.yaml"
sim = spen.from_yaml(config_path, seed=20260510)

InvA, AFinal = sim.get_InvA()
InvA = InvA.to(torch.complex64)
AFinal = AFinal.to(torch.complex64)

corrupted_data, phase_map, good_lr_image, sim_meta = sim.sim(
    img.unsqueeze(0),
    return_phase_map=True,
    return_good_lr_image=True,
    return_metadata=True,
)
corrupted_data = corrupted_data.to(torch.complex64)
good_lr_image = good_lr_image.to(torch.complex64)
phase_map = phase_map.to(torch.float32)

# The scanner-like profile outputs 96x96 data. Resize the displayed
# reference image only for visual comparison with reconstructions.
img_recon_ref = F.interpolate(
    img[None, None].float(),
    size=corrupted_data.shape[1:],
    mode="bilinear",
    align_corners=False,
).squeeze(0).squeeze(0)
img_model = img_recon_ref.to(torch.complex64)

print("config        :", config_path)
print("InvA          :", tuple(InvA.shape),          InvA.dtype)
print("AFinal        :", tuple(AFinal.shape),        AFinal.dtype)
print("corrupted_data:", tuple(corrupted_data.shape), corrupted_data.dtype)
print("phase_map     :", tuple(phase_map.shape),     phase_map.dtype)
print("good_lr_image :", tuple(good_lr_image.shape), good_lr_image.dtype)
print("noise sample  :", sim_meta["noise"])
print("segment shift :", sim_meta["sampled"]["segment_shift_cm"])


## 3. Inspect the simulation outputs

Quick visual check: original image, resized reconstruction reference,
corrupted SPEN magnitude, clean low-resolution magnitude, estimated phase,
and true phase. The estimate differs from truth when the YAML profile uses
`estimate_error_std_rad`, which is useful when training models that must be
robust to imperfect phase estimates.


In [ ]:
fig, axs = plt.subplots(1, 6, figsize=(22, 4))

axs[0].imshow(to_plot(img), cmap="gray");                          axs[0].set_title("Original image")
axs[1].imshow(to_plot(img_recon_ref), cmap="gray");                axs[1].set_title("Resized reference")
axs[2].imshow(to_plot(torch.abs(corrupted_data)[0]), cmap="gray"); axs[2].set_title("|corrupted_data|")
axs[3].imshow(to_plot(torch.abs(good_lr_image)[0]), cmap="gray");  axs[3].set_title("|good_lr_image|")
axs[4].imshow(to_plot(phase_map[0]), cmap="twilight");             axs[4].set_title("phase estimate")
axs[5].imshow(to_plot(sim_meta["phase_map_true"][0]), cmap="twilight"); axs[5].set_title("phase truth")

for ax in axs:
    ax.axis("off")
plt.tight_layout()
plt.show()


## 4. Reconstruct — order matters: phase correction first, then `InvA`

The corrupting phase is a multiplicative factor that lives in the image
domain along the readout direction. `InvA` is a matrix that operates
along the PE direction.

If we apply `InvA` first, the per-row phase corruption gets mixed across
all PE rows by the matrix multiplication and can no longer be removed by
a simple `exp(-i*phi)` multiply.

If we apply phase correction first, one `exp(-i*phi)` per even row in the
ROFFT image, then `InvA` along PE, the corruption is reduced before
mixing.

### Step 4a - apply phase correction to even rows


In [ ]:
# Even-row phase correction in the ROFFT image domain.
even_rows = corrupted_data[:, 1::2, :].clone()
even_rows = even_rows * torch.exp(-1j * phase_map)

corrupted_phase_corrected = corrupted_data.clone()
corrupted_phase_corrected[:, 1::2, :] = even_rows


### Step 4b — compare both orders side by side

* **`InvA × corrupted`** (wrong order): residual ghost-like artifacts.
* **`InvA × phase-corrected`** (right order): clean reconstruction.


In [ ]:
recon_wrong_order = torch.matmul(InvA, corrupted_data)              # InvA first, no phase corr
recon_right_order = torch.matmul(InvA, corrupted_phase_corrected)   # phase corr first, then InvA

fig, axs = plt.subplots(1, 5, figsize=(20, 4.4))

axs[0].imshow(to_plot(img_recon_ref), cmap="gray");                  axs[0].set_title("Reference")
axs[1].imshow(to_plot(torch.abs(corrupted_data)[0]), cmap="gray");  axs[1].set_title("|corrupted_data|")
axs[2].imshow(to_plot(torch.abs(recon_wrong_order)[0]), cmap="gray");
axs[2].set_title("InvA x corrupted (wrong order)")
axs[3].imshow(to_plot(torch.abs(recon_right_order)[0]), cmap="gray");
axs[3].set_title("InvA x phase-corrected (right order)")
axs[4].imshow(to_plot(phase_map[0]), cmap="twilight");              axs[4].set_title("phase estimate")

for ax in axs:
    ax.axis("off")
plt.tight_layout()
plt.show()


## 5. Why doesn't the right-order reconstruction perfectly match the original?

Even with the correct ordering, the reconstructed image is similar to but
**not pixel-identical to** the resized reference. There are four
independent reasons:

### 5.1 `InvA` is a weighted adjoint, not a true matrix inverse

Looking at `spenpy.core.matrix.calcInvA`, the matrix returned is

```python
AGaussWeighted = AFinal * GaussWeight
InvA = AGaussWeighted.conj().t()
```

So `InvA @ AFinal` is not the identity. It is a Gaussian-smoothed,
band-limited approximation of identity. This is a deliberate design
choice: the true inverse of `AFinal` is ill-conditioned, and the weighted
adjoint is a stable, fast reconstructor at the cost of smoothing.

### 5.2 The returned `phase_map` is an estimate

The YAML profile can set `estimate_error_std_rad`, so the returned phase
estimate is not exactly `sim_meta["phase_map_true"]`. This models the
imperfection of real phase estimation.

### 5.3 The scanner-like profile adds more physics than the old demo

This run may include B0 polynomial warping, shot phase, trajectory shifts,
intensity bias, and k-space noise. These are intentional because the goal
is synthetic training data that generalizes better to scanner data.

### 5.4 Noise

Complex Gaussian noise is added in PE k-space. Its sampled standard
deviation for this run is shown in `sim_meta["noise"]`.

### A clean visual sanity check: `|InvA @ AFinal|`


In [ ]:
invA_AFinal = torch.matmul(InvA, AFinal).abs()

fig, axs = plt.subplots(1, 2, figsize=(11, 4.6))

im0 = axs[0].imshow(invA_AFinal, cmap='magma')
axs[0].set_title("|InvA @ AFinal|  (would be identity if InvA was a true inverse)")
plt.colorbar(im0, ax=axs[0], fraction=0.046, pad=0.04)
axs[0].set_xlabel("PE in"); axs[0].set_ylabel("PE out")

n = invA_AFinal.shape[0]
axs[1].plot(to_plot(invA_AFinal[n // 2]), label=f"row {n//2} (centre)")
axs[1].plot(to_plot(invA_AFinal[n // 4]), label=f"row {n//4}")
axs[1].plot(to_plot(invA_AFinal[3 * n // 4]), label=f"row {3*n//4}")
axs[1].set_title("Profile along three rows: Gaussian smoothing footprint")
axs[1].set_xlabel("PE in")
axs[1].legend()
plt.tight_layout()
plt.show()


## 6. Fast forward path: build degraded data with `AFinal` directly

If you only want a fast SPEN blur for quick experiments, you can skip the
full `spen.sim(...)` simulation and apply `AFinal` directly to an image
that already matches the simulator matrix size.

This path is much faster, but it lacks the richer YAML artifacts. It is
still useful for order-of-operations tests and round-trip experiments.

The cell below:

1. Multiplies `AFinal @ (img_model * 1j)` to get degraded SPEN data.
2. Adds the current even/odd phase estimate to even rows.
3. Demonstrates the correct reconstruction order again: phase correction
   first, then `InvA`.


In [ ]:
# Step 1: forward degradation. img_model is the 96x96 resized reference.
degraded_data = torch.matmul(AFinal, img_model.unsqueeze(0) * 1j)

# Step 2: inject even/odd phase corruption.
blur_data = degraded_data.clone()
blur_data[:, 1::2, :] = degraded_data[:, 1::2, :] * torch.exp(1j * phase_map)

# Step 3: correct phase first, then apply InvA.
corr_data = blur_data.clone()
corr_data[:, 1::2, :] = corr_data[:, 1::2, :] * torch.exp(-1j * phase_map)
recon = torch.matmul(InvA, corr_data)

fig, axs = plt.subplots(1, 5, figsize=(20, 4.4))

axs[0].imshow(to_plot(img_recon_ref), cmap="gray");                 axs[0].set_title("Reference")
axs[1].imshow(to_plot(degraded_data[0].abs()), cmap="gray");        axs[1].set_title("|AFinal x (img*j)|")
axs[2].imshow(to_plot(blur_data[0].abs()), cmap="gray");            axs[2].set_title("|blur_data|")
axs[3].imshow(to_plot(recon[0].abs()), cmap="gray");                axs[3].set_title("|InvA x corrected blur|")
axs[4].imshow(to_plot(phase_map[0].abs()), cmap="gray");            axs[4].set_title("|phase_map|")

for ax in axs:
    ax.axis("off")
plt.tight_layout()
plt.show()



## 7. Order-matters demo on the fast forward path

Same idea as Section 4 but on the `AFinal`-degraded data:

* Stage 1: apply `InvA` *without* phase correction → ghost-like
  artifacts, similar to motion artifacts.
* Stage 2: apply phase correction first, then `InvA` → clean recon.


In [ ]:
# Forward + inject phase.
degraded = torch.matmul(AFinal, img_model.unsqueeze(0) * 1j)
degraded_with_phase = degraded.clone()
degraded_with_phase[:, 1::2, :] *= torch.exp(1j * phase_map)

# Stage 1: wrong order, InvA first.
recon_wrong = torch.matmul(InvA, degraded_with_phase)

# Stage 2: right order, phase correction first.
corr = degraded_with_phase.clone()
corr[:, 1::2, :] *= torch.exp(-1j * phase_map)
recon_right = torch.matmul(InvA, corr)

fig, axs = plt.subplots(1, 5, figsize=(20, 4.4))

axs[0].imshow(to_plot(img_recon_ref), cmap="gray");                 axs[0].set_title("Reference")
axs[1].imshow(to_plot(degraded_with_phase[0].abs()), cmap="gray"); axs[1].set_title("|degraded + phase|")
axs[2].imshow(to_plot(recon_wrong[0].abs()), cmap="gray");         axs[2].set_title("InvA x degraded_with_phase (wrong order)")
axs[3].imshow(to_plot(recon_right[0].abs()), cmap="gray");         axs[3].set_title("InvA x phase-corrected (right order)")
axs[4].imshow(to_plot(phase_map[0].abs()), cmap="gray");           axs[4].set_title("|phase_map|")

for ax in axs:
    ax.axis("off")
plt.tight_layout()
plt.show()


## Take-aways

1. **Order matters.** Phase correction is a per-row image-domain
   operation, while `InvA` is a matrix multiplication along the PE
   direction. Applying `InvA` first scrambles the row-wise corruption
   across all PE rows.
2. **`InvA` is the weighted adjoint, not the true inverse.** Even with
   perfect phase correction, reconstruction is a Gaussian-smoothed
   approximation of the reference.
3. **The `phase_map` returned by `sim` is an estimate.** The truth is in
   `sim_meta["phase_map_true"]`; the YAML profile can intentionally add
   estimate error.
4. **YAML profiles are the new training interface.** Use
   `scanner_like.yaml` as the nominal synthetic domain, mix in
   `aggressive_training.yaml` for robustness, and document any new
   scanner-specific profile in `docs/scanner_parameter_notes.md`.
5. For real Bruker PV360 SPEN data, see [`demo/`](demo/), which uses the
   same `InvA` building blocks on actual k-space acquired by `pv360.m`.
